# Clustering Analysis — K-Means vs DBSCAN

Clustering is **unsupervised**: we group samples using only the geometry of the features $X$, with *no* target labels driving the fit. Here we segment a synthetic **customer** dataset two different ways and compare them.

1. **K-Means** — partitions the data into a *pre-chosen* number of round clusters $k$. We find a good $k$ with the **elbow method** (inertia) and the **silhouette score**.
2. **DBSCAN** — grows clusters from dense regions, discovers the number of clusters *on its own*, handles arbitrary shapes, and flags low-density points as **noise** (label `-1`).

If the data happens to carry a ground-truth grouping, we treat it as a *held-out check only* — it never enters the clustering itself.

In [ ]:
import numpy as np                                  # numerical arrays / RNG
import pandas as pd                                 # tabular convenience
import matplotlib.pyplot as plt                     # plotting
import seaborn as sns                               # nicer default styling

from sklearn.datasets import make_blobs             # synthesize clear customer segments
from sklearn.preprocessing import StandardScaler    # scale features (K-Means/DBSCAN are distance-based)
from sklearn.cluster import KMeans, DBSCAN          # the two clustering algorithms
from sklearn.metrics import silhouette_score        # internal validation metric (no labels needed)
from sklearn.neighbors import NearestNeighbors      # used to pick DBSCAN's eps via a k-distance plot

# --- Reproducibility: seed every source of randomness we touch ---
SEED = 42
rng = np.random.default_rng(SEED)   # modern NumPy Generator, used for the noise points below
np.random.seed(SEED)                # legacy global RNG, seeded as a safe default

sns.set_theme(style="whitegrid")    # consistent, clean look for every plot
plt.rcParams["figure.dpi"] = 100

## 1. Synthesize a 2-D customer dataset

We imagine two behavioural features per customer, e.g. **annual spend** vs **visit frequency**. Real segments tend to form compact groups, so we draw them with `make_blobs` (four Gaussian clusters of differing tightness). Then we sprinkle in a handful of **random noise customers** — atypical accounts that belong to no segment. These noise points are exactly where DBSCAN shines: it will tag them `-1` instead of forcing them into a cluster.

Two dimensions keep everything easy to *see* on a scatter plot; the same code scales to any number of features.

In [ ]:
# Four segment centres placed apart in the (spend, frequency) plane.
centers = [[-5, -5], [0, 5], [5, 0], [6, 8]]

# make_blobs draws isotropic Gaussian blobs. Different cluster_std per centre gives
# tight and loose segments -> a more realistic (and more challenging) mix.
# y_true is the generating segment id; we KEEP IT ASIDE purely for later validation.
X_blobs, y_true = make_blobs(
    n_samples=[160, 140, 150, 120],   # per-cluster sizes -> uneven, like real segments
    centers=centers,
    cluster_std=[0.8, 1.4, 0.7, 1.1], # varied spread
    random_state=SEED,
)

# Add ~25 uniform 'noise customers' spread across the whole data range. They belong to
# no blob, so any honest clusterer should treat them as outliers.
n_noise = 25
lo = X_blobs.min(axis=0) - 1.0        # a little beyond the data on each axis
hi = X_blobs.max(axis=0) + 1.0
X_extra = rng.uniform(low=lo, high=hi, size=(n_noise, 2))   # (25, 2) scattered points

# Stack blobs + noise into one raw feature matrix. y_extra = -1 marks 'true noise'
# (again, only for our own bookkeeping / validation, never fed to the models).
X_raw = np.vstack([X_blobs, X_extra])
y_extra = np.full(n_noise, -1)
y_true_full = np.concatenate([y_true, y_extra])

print(f"dataset shape: {X_raw.shape}   (blobs={len(X_blobs)}, noise={n_noise})")

# --- Standardize: subtract the mean, divide by std, per feature ---
# K-Means and DBSCAN both rely on Euclidean distance, so a feature measured on a larger
# numeric scale would dominate the distance. StandardScaler puts every feature on equal
# footing (mean 0, std 1). We fit on all data because clustering is unsupervised (no split).
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Peek at the standardized data. Colour the noise points distinctly so we can see them.
plt.figure(figsize=(6, 5))
is_noise = y_true_full == -1
plt.scatter(X[~is_noise, 0], X[~is_noise, 1], c=y_true_full[~is_noise],
            cmap="viridis", s=18, edgecolor="k", linewidth=0.2, label="segment customers")
plt.scatter(X[is_noise, 0], X[is_noise, 1], c="red", marker="x", s=40, label="true noise")
plt.title("Standardized customer data (colour = generating segment)")
plt.xlabel("feature 1 (scaled)"); plt.ylabel("feature 2 (scaled)")
plt.legend(loc="best", fontsize=8)
plt.show()

## 2. K-Means — inertia (WCSS) and the elbow method

K-Means partitions the $n$ points into $k$ clusters $\{C_1,\dots,C_k\}$ with centroids $\mu_j$, minimizing the **within-cluster sum of squares** (WCSS), which scikit-learn reports as `inertia_`:

$$\text{WCSS}(k) \;=\; \sum_{j=1}^{k}\; \sum_{x \in C_j} \lVert x - \mu_j \rVert^2$$

Inertia measures how tight the clusters are. It **always decreases** as $k$ grows (more centroids can hug the data), reaching $0$ when $k=n$. So we cannot just minimize it. Instead we look for the **elbow**: the $k$ after which adding clusters buys only a marginal drop in WCSS. Before the elbow each new cluster explains real structure; after it, we are just splitting already-tight groups.

### The silhouette coefficient

The elbow is a visual heuristic, so we cross-check it with the **silhouette score**, which needs no labels. For a point $i$, let $a(i)$ be its mean distance to the *other points in its own cluster* (cohesion) and $b(i)$ the mean distance to the points of the *nearest other cluster* (separation):

$$s(i) \;=\; \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}} \;\in\; [-1, 1]$$

$s(i)\approx 1$ means the point sits comfortably in its cluster and far from others; $\approx 0$ means it is on a boundary; $<0$ means it is probably mis-assigned. The overall score is the mean of $s(i)$; we prefer the $k$ that **maximizes** it.

In [ ]:
# Sweep k over a sensible range and record both diagnostics for each k.
k_range = range(2, 10)             # silhouette is undefined for k<2, so start at 2
inertias = []                      # WCSS for the elbow curve
silhouettes = []                   # mean silhouette for the silhouette curve

for k in k_range:
    # n_init=10 -> run K-Means 10 times from different seeds and keep the best (lowest inertia).
    # We set it EXPLICITLY so the code is deterministic and free of the 'n_init will change'
    # FutureWarning across sklearn versions. random_state pins the initial centroid draws.
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = km.fit_predict(X)                      # assign every point to a cluster
    inertias.append(km.inertia_)                    # sklearn's WCSS for this k
    silhouettes.append(silhouette_score(X, labels)) # average silhouette over all points

# Tabulate so the numbers are easy to read alongside the plots.
sweep = pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": silhouettes})
print(sweep.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# The k that maximizes the silhouette is our objective 'best k' recommendation.
best_k = int(sweep.loc[sweep["silhouette"].idxmax(), "k"])
print(f"\nk with highest silhouette: {best_k}")

In [ ]:
# Plot the two diagnostics side by side.
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: elbow curve (inertia vs k). Look for the 'bend'. ---
ax[0].plot(list(k_range), inertias, "o-", color="#4c72b0")
ax[0].axvline(best_k, color="grey", ls="--", lw=1)           # mark the chosen k
ax[0].set_title("Elbow method — inertia (WCSS) vs k")
ax[0].set_xlabel("number of clusters k"); ax[0].set_ylabel("inertia (WCSS)")

# --- Right: silhouette score vs k. Look for the peak. ---
ax[1].plot(list(k_range), silhouettes, "o-", color="#dd8452")
ax[1].axvline(best_k, color="grey", ls="--", lw=1)
ax[1].set_title("Silhouette score vs k")
ax[1].set_xlabel("number of clusters k"); ax[1].set_ylabel("mean silhouette (higher = better)")

plt.tight_layout(); plt.show()

**Reading the two plots together.** The elbow curve drops steeply up to the point where the sharp bend occurs and then flattens — each extra cluster past that point barely reduces WCSS. The silhouette curve independently *peaks* at the same value. When the visual elbow and the silhouette maximum agree, we can be confident in the choice. We therefore fix $k$ at the value printed below and fit the final K-Means model.

In [ ]:
# Fit the final K-Means with the chosen k (again n_init explicit, seed pinned).
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=SEED)
kmeans_labels = kmeans.fit_predict(X)

print(f"final K-Means: k = {best_k}")
print(f"inertia (WCSS): {kmeans.inertia_:.3f}")
print(f"silhouette:     {silhouette_score(X, kmeans_labels):.3f}")

# Note: K-Means always assigns EVERY point to some cluster -> the true noise points get
# folded into whichever centroid is nearest. It has no concept of an outlier.

## 3. DBSCAN — density-based clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) needs no $k$. It has two parameters:

- **`eps`** ($\varepsilon$): the neighbourhood radius around a point.
- **`min_samples`**: how many points must fall within that radius for a region to count as *dense*.

Every point is then classified as:

- **Core point** — has at least `min_samples` neighbours within `eps`. It anchors a cluster.
- **Border point** — within `eps` of a core point but not itself dense. It joins that cluster's edge.
- **Noise point** — neither core nor border. Labelled **`-1`** and left out of every cluster.

Clusters grow by chaining together density-reachable core points, so they can take **arbitrary shapes** (not just round blobs) and outliers are naturally excluded.

### Choosing `eps` — the k-distance intuition

A common heuristic: fix `min_samples` (a rule of thumb is $\approx 2 \times \text{dimensions}$), compute each point's distance to its $k$-th nearest neighbour (with $k = \text{min\_samples}$), sort those distances ascending, and plot them. Points inside clusters have small k-distances; noise points have large ones. The curve stays flat then bends sharply upward — the **knee** of that curve is a good `eps`: large enough to link genuine neighbours, small enough to exclude the sparse tail.

In [ ]:
# min_samples heuristic: ~2 * n_features. Our data is 2-D, so 4 is a reasonable start.
min_samples = 4

# Distance to each point's k-th nearest neighbour (k = min_samples).
# NearestNeighbors returns the point itself as neighbour 0 (distance 0), so we ask for
# min_samples+1 neighbours and read the LAST column = the min_samples-th true neighbour.
nbrs = NearestNeighbors(n_neighbors=min_samples + 1).fit(X)
dists, _ = nbrs.kneighbors(X)
k_dist = np.sort(dists[:, -1])          # sort those k-distances ascending for the plot

plt.figure(figsize=(6, 4))
plt.plot(k_dist, color="#55a868")
plt.title(f"k-distance plot (k = {min_samples}) — look for the knee")
plt.xlabel("points sorted by distance")
plt.ylabel(f"distance to {min_samples}-th neighbour")
plt.show()

# The flat body of the curve sits below ~0.25 in these standardized units, and the
# curve kicks up sharply after that as we reach the sparse noise points. We pick eps in
# that knee region. (You can read it straight off the plot above.)
eps = 0.25
print(f"chosen eps = {eps},  min_samples = {min_samples}")

In [ ]:
# Fit DBSCAN with the eps/min_samples chosen above. DBSCAN is deterministic given the
# data and parameters, so no random_state is needed.
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X)

# Cluster count excludes the noise label (-1). set() gives the unique labels DBSCAN found.
unique = set(dbscan_labels)
n_clusters = len(unique) - (1 if -1 in unique else 0)
n_noise = int(np.sum(dbscan_labels == -1))

print(f"DBSCAN found {n_clusters} clusters")
print(f"DBSCAN flagged {n_noise} points as noise (label -1)")

# core_sample_indices_ lists the core points; the rest of a cluster are border points.
print(f"core points: {len(dbscan.core_sample_indices_)}  "
      f"(the remaining clustered points are border points)")

## 4. Side-by-side comparison

We now plot the exact same points coloured by K-Means labels vs DBSCAN labels. Watch the noise customers: K-Means absorbs each into its nearest round cluster, while DBSCAN paints them black as `-1`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: K-Means. Every point is coloured; centroids marked with a red X. ---
ax[0].scatter(X[:, 0], X[:, 1], c=kmeans_labels, cmap="viridis",
              s=18, edgecolor="k", linewidth=0.2)
ax[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
              c="red", marker="X", s=180, edgecolor="white", label="centroids")
ax[0].set_title(f"K-Means (k={best_k}) — every point assigned")
ax[0].set_xlabel("feature 1 (scaled)"); ax[0].set_ylabel("feature 2 (scaled)")
ax[0].legend(loc="best", fontsize=8)

# --- Right: DBSCAN. Plot noise (-1) separately in black; clusters in colour. ---
noise_mask = dbscan_labels == -1
ax[1].scatter(X[~noise_mask, 0], X[~noise_mask, 1], c=dbscan_labels[~noise_mask],
              cmap="viridis", s=18, edgecolor="k", linewidth=0.2)
ax[1].scatter(X[noise_mask, 0], X[noise_mask, 1], c="black", marker="x",
              s=45, label=f"noise (-1): {n_noise} pts")
ax[1].set_title(f"DBSCAN (eps={eps}) — {n_clusters} clusters + noise")
ax[1].set_xlabel("feature 1 (scaled)"); ax[1].set_ylabel("feature 2 (scaled)")
ax[1].legend(loc="best", fontsize=8)

plt.tight_layout(); plt.show()

## 5. When to prefer each

| | **K-Means** | **DBSCAN** |
|---|---|---|
| Number of clusters | you must choose $k$ up front | discovered automatically |
| Cluster shape | assumes convex, roughly **round** clusters of similar size | finds **arbitrary shapes** |
| Outliers | none — every point is assigned | isolated as **noise (`-1`)** |
| Key knob | $k$ (use elbow + silhouette) | `eps` & `min_samples` (use the k-distance knee) |
| Sensitivity | sensitive to initialization (mitigated by `n_init`) & to scale | very sensitive to `eps`; struggles when clusters have very different densities |

**Rules of thumb.**

- Reach for **K-Means** when you expect a known number of compact, similarly-sized segments and want fast, simple assignments for *every* customer.
- Reach for **DBSCAN** when clusters may be irregularly shaped, when you don't know how many there are, or when you explicitly want to *detect and exclude outliers* — as with the noise customers here.

Both are unsupervised: the generating segment ids were used only to build and eyeball the data, never to fit the models. In practice you would validate the discovered clusters against business meaning or an internal metric like the silhouette score.